Import the required modules and libraries

In [ ]:
#Import the required modules and libraries
import pastaq as pq
import math
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as colors
from collections import defaultdict
from pathlib import Path
import re
from itertools import combinations
import ms_entropy as me
import ast
from dataclasses import dataclass
import networkx as nx
import math
import numpy as np
import json

Import helpers for PASTAQ. Here the helper functions have been added to a .py file called "PASTAQ_helpers" to keep the notebook tidy. 

In [ ]:
import PASTAQ_helpers

Import the code intended to run after PASTAQ's dda_pipeline. The .py script is titled "py_post_dda" in the directory.

In [ ]:
import pq_post_dda as pqpost

Define the input path to the raw neg.ms2 files created by PASTAQ's dda pipeline and the output path for PASTAQ's dda_pipeline.

In [ ]:
output_dir = r"C:/Users/User/Location"

In [ ]:
input_dir = Path('C:/Users/User/Location/pastaq/raw')

In [ ]:
input_files = [{'raw_path': str(input_dir / 'sample1.ms2')},
                {'raw_path': str(input_dir / 'sample2.ms2')},
                {'raw_path': str(input_dir / 'sample3.ms2')},
]

Define the path to the "feature_clusters_annotations.csv" generated by PASTAQ's DDA pipeline

In [ ]:
feature_clusters_annotations_csv = pd.read_csv(r"C:/Users/User/Location/pastaq/quant/feature_clusters_annotations.csv")

Link the MS2 information to the csv file containing the feature clusters, this function has been updated to allow the user to decide if they wish to keep the features with no linked MS2. 

Notes:
If you wish to remove the features that have no linked MS2, please set "MS2_only" to "True". If you wish to keep all the features (including those with no linked MS2), please set "MS2_only" to "False". It will default to "False" if you do not change it.

In [ ]:
combined_multiple_samples = pqpost.combine_multiple_samples(feature_clusters_annotations_csv, input_files, output_dir)

Specify the input pathways for the "neg.features" files created by PASTAQs' DDA pipeline.

In [ ]:
input_dir = Path('C:/Users/User/Location/pastaq/features')

In [ ]:
input_files = [{'raw_path': str(input_dir / 'sample1.features')},
                {'raw_path': str(input_dir / 'sample2.features')},
                {'raw_path': str(input_dir / 'sample3.features')},
]

Link the MS1 feature-level information to the large dataframe now containing both the MS2 information and the feature cluster-level information for all of your samples.

This is necessary as the "feature_clusters_annotations.csv" file does not contain information such as intensity or m/z by default. The feature-level information for each individual sample can also be found in the "[sample name]features.csv" files generated by PASTAQ's dda pipeline. 

In [ ]:
linked_features = pqpost.link_features(combined_multiple_samples=combined_multiple_samples, input_files=input_files, output_dir=output_dir)

Run the function to remove noise and average out MS2 values for MS1 peaks with multiple linked MS2 values.

This function works per ONE "peak_id" within ONE "file_id", it does not calculate an average MS2 result for one feature overall.

This function will only keep MS2 spectra that are present in at least a certain proportion (default min_fraction=0.5) of the MS2 spectra, and averages out the raw spectra that are within a certain ppm distance from each other (default ppm_tolerance=10). This function also calculates the entropy similarity between the raw spectra before averaging is performed.

If an MS1 peaks has only one linked MS2 spectrum, this spectrum will not be altered.

In [ ]:
average_linked_ms2 = pqpost.average_linked_ms2(linked_features)

Aggegrate your data into their feature-level output.

Peaks with linked MS2 are retained separately but peaks with no linked MS2 are collapsed into a single feature-level result.

This is done to remove redundant "peak_ids" (peaks that belong to the same feature will have the same precursor, total and average information on the MS1 level). This speeds up later processing without losing information.



In [ ]:
aggregated_features = pqpost.aggregate_features(average_linked_ms2)

Load and read your Excel file containing your internal standards. 

It should be in the format in the example Excel file supplied.

In [ ]:
#Load file with list of internal standards
IS_file = r"C:/Users/User/Location/Internal_Standards_File.xlsx"
IS_data = pd.read_excel(IS_file)

Load your MSP file containing the information needed for lipid identification. This workflow was tested using the following MSP files: "MsMs_Positive_LabOnly_AllDirectories.msp", "Msp_2026_07_20_14_51_17_AlignmentResult_2026_07_07_11_03_22.msp", "MSDIAL-TandemMassSpectralAtlas-VS69-Pos.msp", "Msp_2026_08_28_12_07_56_AlignmentResult_2026_07_08_13_31_05", MSMS-Public_all-neg-VS19, and MSMS-Public_experimentspectra-neg-VS19.

In [ ]:
#Load msp file
msp_file = r"C:\Users\diego.DESKTOP-7OSFK5B\Documents\MSc_Research_Project_2\Code\MSPs\Msp_2026_08_28_12_07_56_AlignmentResult_2026_07_08_13_31_05.msp"

Read your MSP file.

In [ ]:
msp_data = pqpost.read_msp_file(msp_file)

Identify annotation matches (internal standard or lipid/ metabolite) for features. 

"unknown_classes" refers to annotations that are present in some MSP libraries, but do not have an actual lipid identity provided (e.g. "RIKEN P-VS1 ID-1760 from Mouse_Muscle_WT_CTX3_Ctr"), setting "unknown_classes" to "False" will remove these annotations.

"Unidentified" refers to unidentified features, the code will default to "True". Please change "unidentified=True" to "unidentified=False" if you do not wish to keep unidentified features.

"score" refers to the weighted difference between the feature and the annotation, hence a lower score means a smaller difference or better match between the feature in the sample and the annotation from the files.

The scoring works as follows:
1) Internal standards: filtered based on both precursor m/z and precursor RT then scored based on the weighted formula incorporating the precursor m/z and precursor RT.
2) MSP annotations: features with an m/z distance above the "mz_tolerance" will not be considered for feature annotation, features are then scored based on MS1 (if only MS1 information is available) or both MS1 and MS2. These scores are also weighted - MS1 m/z distance and MS2 spectral difference are given equal weights, with MS1 only features being penalized to avoid artificially inflated scores. 

In [ ]:
matches = pqpost.identify_matches(
    aggregated_features,
    msp_data,
    IS_data,
    mz_tolerance=0.025,
    IS_mz_tolerance=0.014,
    IS_rt_tolerance=60.0,
    unidentified=True,
    unknown_classes=False
)


Aggregated the annotated features into one result per feature. 

This function will select the peak with the highest entropy similarity as the representative peak for the feature if a feature contains multiple "peaks_id"s with linked MS2. 

In [ ]:
aggregated_matches = pqpost.aggregate_matches(matches)

Normalize your data to either your internal standards (for identified clusters) or the total ion current (for unidentified clusters).

In [ ]:
normalized_samples = pqpost.normalize_features(
    aggregated_matches,
    rt_tolerance=300,
    normalize_to_group=True,
    normalize_to_class=True,
    IS_normalization_only=True
)

Declare which "file_id" belongs to your blank, here the "file_id" that corresponds to the blank is "b1". Your "file_id" is the stem of the mzML file you submitted to PASTAQ's dda_pipeline (e.g. for this example, the file was called "b1.mzML").

This code was tested using only one blank, hence it may behave unexpectedly if you specify multiple blanks.

In [ ]:
blank = [
    feature
    for feature in normalized_samples
    if feature["file_id"] == "b1"
]

Determine the overall annotations for the group, flag any clusters that you may later wish to remove or modify, and calculate summary statistics such as the coefficient of variation and 95% CI.

In [ ]:
annotation_results = pqpost.determine_group_annotations(
    normalized_samples,
    blank=blank,
    CV_limit=30,
    blank_rt_tolerance=300,
    mad_threshold=4
)

Convert your annotation results to a PANDAS dataframe and convert the results into a serializable JSON file. This is to save your results, you can use the visualization functions without first saving your results as a JSON file. 

In [ ]:
annotation_results_df = pd.DataFrame(annotation_results)

In [ ]:
def make_json_serializable(obj):

    # ---------------------------------------------------------
    # Pandas missing value
    # ---------------------------------------------------------
    if obj is pd.NA:
        return None

    # ---------------------------------------------------------
    # Dictionaries
    # ---------------------------------------------------------
    if isinstance(obj, dict):
        return {
            key: make_json_serializable(value)
            for key, value in obj.items()
        }

    # ---------------------------------------------------------
    # Lists / tuples
    # ---------------------------------------------------------
    if isinstance(obj, (list, tuple)):
        return [
            make_json_serializable(value)
            for value in obj
        ]

    # ---------------------------------------------------------
    # NumPy arrays
    # ---------------------------------------------------------
    if isinstance(obj, np.ndarray):
        return make_json_serializable(obj.tolist())

    # ---------------------------------------------------------
    # NumPy floating-point values
    # ---------------------------------------------------------
    if isinstance(obj, np.floating):
        if not np.isfinite(obj):
            return None
        return obj.item()

    # ---------------------------------------------------------
    # Python floats
    # ---------------------------------------------------------
    if isinstance(obj, float):
        if not math.isfinite(obj):
            return None
        return obj

    # ---------------------------------------------------------
    # NumPy integers
    # ---------------------------------------------------------
    if isinstance(obj, np.integer):
        return obj.item()

    # ---------------------------------------------------------
    # NumPy booleans
    # ---------------------------------------------------------
    if isinstance(obj, np.bool_):
        return bool(obj)

    return obj

In [ ]:
def save_annotation_results_json(annotation_results, json_path):

    records = annotation_results.to_dict(orient="records")

    records = make_json_serializable(records)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(
            records,
            f,
            indent=2,
            ensure_ascii=False,
            allow_nan=False
        )

In [ ]:
save_annotation_results_json(
    annotation_results_df,
    "annotation_results_df.json"
)

Run the code to allow you to visualize the annotation results.
This allows interactive visualization of the results plotted as precursor m/z vs retention time.
Use rectangle zoom in to zoom in to a selected region. Clicking on a cluster dot will display the annotation information.

Visualizing all annotations in all samples at the same time can get very confusing, very quickly, hence you can also filter by any of the below or any combination of the below:
    a) sample group(s)
    b) annotations above a specified average intensity
    c) class(es)
    d) MS2 status

In [ ]:
def visualize_annotation_results(annotation_results, classes=None, groups=None, ncols=2):
    """Plot selected annotations as precursor m/z versus retention time.

    Click on a cluster dot to display its retention time, m/z, MS2 status,
    average intensity, intenisity 95% confidence intervals, annotation name, and saturation.

    Parameters
    ----------
    annotation_results : list of dict
        Overall annotation results, one entry per group and cluster_id.
    classes : str or iterable of str, optional
        Lipid class or classes to display, for example ["PC"] or ["CL", "Unidentified"].
        By default, all classes are displayed.
    groups : str or iterable of str, optional
        Biological groups or groups to display. By default, all groups are displayed.
    ncols : int, default=2
        Number of biological-group subplots per row.

    Returns
    -------
    tuple
        Matplotlib figure and axes. ``axes`` is always a one-dimensional
        NumPy array, including when only one group is selected.
    """
    class_colors = {
        "PC": "blue", "PE": "green", "PG": "orange", "PI": "purple",
        "PS": "red", "PA": "brown", "CL": "cyan", "SM": "magenta",
        "Cer": "yellow", "HexCer": "pink", "DG": "gray", "TG": "black",
        "FA": "lightblue", "LPA": "lightgreen", "LPC": "lightcoral",
        "LPE": "lightyellow", "LPG": "lightgray", "LPI": "lightcyan",
        "LPS": "lightpink", "Unidentified": "darkgray", "Unclear": "darkred",
    }

    def as_filter_set(value):
        if value is None:
            return None
        if isinstance(value, str):
            return {value}
        return {str(item) for item in value}

    def get_lipid_class(annotation_name):
        if annotation_name is None:
            return "Unclear"
        annotation_name = str(annotation_name).strip()
        if not annotation_name:
            return "Unclear"
        for lipid_class in sorted(class_colors, key=len, reverse=True):
            if annotation_name == lipid_class or annotation_name.startswith(lipid_class):
                return lipid_class
        return "Other"

    def display_value(value, formatter=None):
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return "NA"
        return formatter(value) if formatter else str(value)

    selected_classes = as_filter_set(classes)
    selected_groups = as_filter_set(groups)
    points = []

    for result in annotation_results:
        group = str(result.get("group", "Unknown"))
        if selected_groups is not None and group not in selected_groups:
            continue
        annotation_name = result.get("identified_annotation_name")
        lipid_class = get_lipid_class(annotation_name)
        if selected_classes is not None and lipid_class not in selected_classes:
            continue
        try:
            mz = float(result["avg_precursor_mz"])
            rt = float(result["avg_precursor_rt"])
        except (KeyError, TypeError, ValueError):
            continue
        if not np.isfinite(mz) or not np.isfinite(rt):
            continue
        try:
            n_replicates = max(float(result.get("n_replicates", 1)), 1.0)
        except (TypeError, ValueError):
            n_replicates = 1.0
        points.append({
            "group": group,
            "mz": mz,
            "rt": rt,
            "class": lipid_class,
            "color": class_colors.get(lipid_class, "lightgray"),
            "size": 35 + 20 * np.sqrt(n_replicates),
            "label": str(result.get("cluster_id", "?")),
            "has_ms2": bool(result.get("MS2_replicates")),
            "mean_intensity": result.get("mean_intensity"),
            "intensity_ci_lower": result.get("intensity_ci_lower"),
            "intensity_ci_upper": result.get("intensity_ci_upper"),
            "annotation_name": annotation_name,
            "saturation": result.get("identified_annotation_saturation"),
        })

    group_names = list(dict.fromkeys(point["group"] for point in points)) or ["No matching results"]
    ncols = max(int(ncols), 1)
    nrows = int(np.ceil(len(group_names) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(7 * ncols, 5.5 * nrows),
        sharex=True, sharey=True, squeeze=False,
    )
    axes = axes.ravel()
    scatter_artists = []
    selected_annotations = {}

    for ax, group_name in zip(axes, group_names):
        group_points = [point for point in points if point["group"] == group_name]
        for lipid_class in dict.fromkeys(point["class"] for point in group_points):
            class_points = [point for point in group_points if point["class"] == lipid_class]
            scatter = ax.scatter(
                [point["mz"] for point in class_points],
                [point["rt"] for point in class_points],
                s=[point["size"] for point in class_points],
                c=class_points[0]["color"], label=lipid_class,
                alpha=0.8, edgecolors="black", linewidths=0.5,
                picker=True,
            )
            scatter._cluster_metadata = class_points
            scatter_artists.append(scatter)
        for point in group_points:
            ax.annotate(point["label"], (point["mz"], point["rt"]),
                        xytext=(4, 4), textcoords="offset points", fontsize=8)
        selected_annotations[ax] = ax.annotate(
            "", xy=(0, 0), xytext=(12, 12), textcoords="offset points",
            bbox={"boxstyle": "round,pad=0.4", "fc": "white", "ec": "black", "alpha": 0.9},
            arrowprops={"arrowstyle": "->"}, visible=False,
        )
        ax.set_title(group_name)
        ax.grid(True, linestyle=":", alpha=0.35)
        if group_points:
            ax.legend(title="Lipid class", fontsize=8)
        else:
            ax.text(0.5, 0.5, "No matching results", ha="center", va="center", transform=ax.transAxes)

    def on_cluster_click(event):
        if event.inaxes is None:
            return
        for scatter in scatter_artists:
            if scatter.axes is not event.inaxes:
                continue
            contains, details = scatter.contains(event)
            if not contains or not details.get("ind"):
                continue
            point = scatter._cluster_metadata[details["ind"][0]]
            annotation = selected_annotations[event.inaxes]
            annotation.xy = (point["mz"], point["rt"])
            annotation.set_text(
                f"Cluster: {point['label']}\n"
                f"RT: {display_value(point['rt'], lambda value: f'{value:.2f}')}\n"
                f"m/z: {display_value(point['mz'], lambda value: f'{value:.5f}')}\n"
                f"Linked MS2: {'Yes' if point['has_ms2'] else 'No'}\n"
                f"Average intensity: {display_value(point['mean_intensity'], lambda value: f'{float(value):.3g}')}\n"
                f"95% CI: "
                f"{display_value(point['intensity_ci_lower'], lambda value: f'{float(value):.3g}')} - "
                f"{display_value(point['intensity_ci_upper'], lambda value: f'{float(value):.3g}')}\n"
                f"Annotation: {display_value(point['annotation_name'])}\n"
                f"Saturation: {display_value(point['saturation'])}"
            )
            annotation.set_visible(True)
            event.inaxes.figure.canvas.draw_idle()
            break

    fig.canvas.mpl_connect("button_press_event", on_cluster_click)

    for ax in axes[len(group_names):]:
        ax.set_visible(False)
    for ax in axes[:len(group_names)]:
        ax.set_xlabel("Precursor m/z")
        ax.set_ylabel("Retention time")

    class_title = "all classes" if selected_classes is None else ", ".join(sorted(selected_classes))
    fig.suptitle(f"Annotation results: {class_title}", y=1.02)
    fig.tight_layout()
    return fig, axes[:len(group_names)]

In [ ]:
_visualize_annotation_results = visualize_annotation_results

def visualize_annotation_results(
    annotation_results,
    classes=None,
    groups=None,
    ncols=2,
    has_ms2=None,
    min_average_intensity=None,
 ):
    """Apply optional MS2 and intensity filters before plotting."""
    filtered_results = []

    for result in annotation_results:
        if has_ms2 is not None:
            result_has_ms2 = bool(result.get("MS2_replicates"))
            if result_has_ms2 != has_ms2:
                continue

        if min_average_intensity is not None:
            intensity = result.get("mean_intensity")
            try:
                intensity = float(intensity)
            except (TypeError, ValueError):
                continue
            if not np.isfinite(intensity) or intensity < min_average_intensity:
                continue

        filtered_results.append(result)

    return _visualize_annotation_results(
        filtered_results,
        classes=classes,
        groups=groups,
        ncols=ncols,
    )

Ensure ipympl is installed in the same Python environment as the notebook kernel, otherwise the image generated will not be static and not interactive.

In [ ]:
# ipympl has to be installed in the same Python environment as the notebook kernel
# otherwise the interactive backend will not work. 
# If you are using a virtual environment, make sure to install ipympl in that environment. 
# If you are using JupyterLab, you may also need to install the jupyter-matplotlib extension.
import sys

try:
    import ipympl
    %matplotlib widget
    print("Interactive Matplotlib backend enabled.")
except Exception as error:
    print(f"Interactive backend unavailable: {error}")
    print("Install ipympl into this notebook interpreter, then restart the kernel:")
    print(f'  "{sys.executable}" -m pip install ipympl')

Visualize your annotated results.

In [ ]:
fig, axes = visualize_annotation_results(
    annotation_results,
    classes=["PC"],
    groups=["Kar"],
    has_ms2=True,
    min_average_intensity=1000,
)

fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = "top"
fig.canvas.header_visible = True
fig.canvas.footer_visible = True
fig.canvas

Import py4cytoscape and ping it to ensure it is connected. Cytocscape should be downloaded and running to use the following code. The marker network connects points based on the weighted entropy score. The transformations used here are those used by the Cytoscape app, MettaNetter_2. 

In [ ]:
import py4cytoscape as p4c
p4c.cytoscape_ping()

In [ ]:
style_name = "Marker Style"

p4c.create_visual_style(style_name)

In [ ]:
CLASS_COLORS = {
    "CL": "#e31818",
    "CAR": "#e3b418",
    "CE": "#e3cb18",
    "Cer":"#e56612",
    "HexCer": "brown",
    "Hex2Cer": "brown",
    "Hex3Cer": "brown", 
    "FA": "#103913",
    "LPC": "#1841e3",
    "LPE": "#0e2a9a",
    "LPI": "#091c69",
    "LPS": "#030922",
    "MG": "#e318d2",
    "DG": "#981094",
    "TG": "#4b0a3c",
    "PC": "#bb29f6",
    "PE": "#8a1cb5",
    "PS": "#5e137c",
    "PI": "#2b0839",
    "PG": "#1d0626",
    "PA": "#bc76d8",
    "SM": "#2ccf24",
    "Unknown": "#3f3535",
}

OTHER_COLORS = [
    "#8dd3c7",
    "#ffffb3",
    "#bebada",
    "#fb8072",
    "#80b1d3",
    "#fdb462",
    "#b3de69",
    "#fccde5",
    "#d9d9d9",
    "#9f83a0",
]

EDGE_COLORS = {
        "Hydrogenation (+H2)": "#17B18F",
        "CH2 addition": "#D15609",
        "Elongation (+C2H4)": "#07420F",
        "Oxidation (+O)": "#D62728",
        "Dioxidation (+2O)": "#811414",
        "Trioxidation (+3O)": "#420909",
        "Hydration (+H2O)": "#0F79CF",
        "Aminotransferase": "#e318c1",
        "Tertiary Amine (+N)": "#052C08",
        "Secondary Amine (+NH)": "#05740C",
        "Primary Amine (+NH2)": "#22e318",
        "+CO": "#e31818",
        "+C2H4": "#e3b418",
        "+C4H8": "#e3cb18",
        "+HPO4": "#BF6313",
        "+C2H2O": "#0A2B5E",
        "+CH2N2": "#13BF2D",
        "+C2H4N":  "#13BF2D",
        "+CO2": "#BF1313",
        "+CH2ON":  "#13BF2D",
        "+CHO2": "#1BB7DA",
        "+CO2H2": "#1BB7DA",
        "C2O2": "#1BB7DA",
        "Poly Amine Reaction A": "#96B1DA",
        "+C2H3NO": "#13BF2D",
        "+C3H5O": "#1BB7DA",
        "+C3H7N":  "#13BF2D",
        "+C2H3O2": "#1BB7DA",
        "+CH3N2O":  "#13BF2D",
        "+C2H4O2": "#1BB7DA",
        "Pyrophosphate (+PP)": "#1358BF",
        "+CH2O3": "#1BB7DA",
        "+C5H7": "#1358BF",
        "+C3H5NO":  "#13BF2D",
        "+SO3": "#E55E24",
        "+HPO3": "#CC15CF",
        "+HSO3": "#E55E24",
        "+C4H4O2": "#1BB7DA",
        "+C3H2O3": "#1BB7DA",
        "+C5H12N": "#13BF2D",
        "+C3H5NO2":  "#13BF2D",
        "+C5H7NO":  "#13BF2D",
        "+C5H9NO": "#13BF2D",
        "+C4H7NO2":  "#13BF2D",
        "+C4H6O3": "#1358BF",
        "+C3H5NOS":  "#13BF2D",
        "+C4H4N3O":  "#13BF2D",
        "+C4H3N2O2": "#1358BF",
        "+C6H11NO": "#1358BF",
        "+C4H6N2O2": "#1358BF",
        "+C5H10N2O": "#1358BF",
        "+C4H5NO3": "#1358BF",
        "+C5H5N2O2":  "#13BF2D",
        "+C5H8N2O2":  "#13BF2D",
        "+C6H12N2O":  "#13BF2D",
        "Poly Amine Reaction B": "#96B1DA",
        "+C5H7NO3":  "#13BF2D",
        "+C5H9NOS":  "#13BF2D",
        "+C5H8O4": "#1BB7DA",
        "+C5H4N5":  "#13BF2D",
        "+C6H7N3O":  "#13BF2D",
        "+C9H9NO":  "#13BF2D",
        "+C5H4N5O":  "#13BF2D",
        "+C6H12N4O":  "#13BF2D",
        "+H3O6P2": "#BF6313",
        "+C6H10O5": "#1BB7DA",
        "+C9H9NO2":  "#13BF2D",
        "+C3H7O6P": "#CC15CF",
        "+C6H8O6":"#1BB7DA",
        "+C6H10O6":  "#13BF2D",
        "+C11H10N2O":  "#13BF2D",
        "+C6H10N2O3S2": "#E55E24",
        "+C10H12N2O4":  "#13BF2D",
        "+C9H10N2O5":  "#13BF2D",
        "+C10H14N2O2S": "#E55E24",
        "+C8H8NO5P":  "#13BF2D",
        "+C16H30O":"#1BB7DA",
        "+C6H11O8P":  "#13BF2D",
        "+C10H15N2O3S":  "#E55E24",
        "+C10H11N5O3":  "#13BF2D",
        "+C10H11N5O4":  "#13BF2D",
        "+C18H35NO":  "#13BF2D",
        "+C10H15N3O5S": "#E55E24",
        "+C10H13N2O7P":  "#13BF2D",
        "+C9H12N3O7P":  "#13BF2D", 
        "+C9H11N2O8P":  "#13BF2D",
        "+C9H14N3O8P":  "#13BF2D",
        "+C10H12N5O6P":  "#13BF2D",
        "+C12H20O11":"#1BB7DA",
        "+C10H12N5O7P":  "#13BF2D",
        "+C10H14N2O10P2":  "#13BF2D",
        "+C9H13N3O10P2":  "#13BF2D",
        "+C9H12N2O11P2":  "#13BF2D",
        "+C10H13N5O9P2":  "#13BF2D",
        "Poly Amine Reaction C": "#96B1DA",
        "+C10H13N5O10P2":  "#13BF2D",
        "+C18H30O15": "#1BB7DA",
        "+C17H30N7O10P3S": "#E55E24",
        "+C19H31N6O14P3S": "#E55E24",
        "+C27H47N9O9S2": "#E55E24",
        "+C20H34N7O14P3S": "#E55E24",
        "+C21H34N7O15P3S": "#E55E24",
        "Succinyl CoA Synthetase": "#96B1DA",
        "+C15H9I4NO3": "#24E5BB",
        "+C21H32N7O16P3S": "#E55E24",
        "+C21H35N7O16P3S": "#E55E24",

        None: "#B0B0B0",
    }

In [ ]:
def get_node_colors(G):
    """
    Create a colour mapping for all lipid classes in G.

    Main lipid classes use predefined colours.
    Other classes are automatically assigned colours
    from OTHER_COLORS.
    """

    # Get all lipid classes present in the network
    classes = {
        data.get("lipid_class")
        for _, data in G.nodes(data=True)
        if data.get("lipid_class") is not None
    }

    # Start with predefined colours
    node_colors = CLASS_COLORS.copy()

    # Classes without a predefined colour
    other_classes = sorted(
        classes - set(node_colors)
    )

    # Assign colours to other classes
    for i, lipid_class in enumerate(other_classes):
        node_colors[lipid_class] = OTHER_COLORS[
            i % len(OTHER_COLORS)
        ]

    return node_colors

In [ ]:
#1) build the graph for the marker network
def show_marker_network(
    G,
    network_name="Marker Network",
    style_name="Marker Style",
):
    """
    Send a NetworkX graph to Cytoscape and apply a visual style.
    """

    # Generate colours based on classes actually present
    node_colors = get_node_colors(G)

    # Import the graph
    p4c.create_network_from_networkx(
        G,
        title=network_name
    )

    # Create/update the style
    style_marker_network(
        style_name=style_name,
        node_colors=node_colors
    )

    # Apply style
    p4c.set_visual_style(style_name)

    # Apply layout
    p4c.layout_network("force-directed")

#2) Define the style of the marker network
def style_marker_network(
    style_name="Marker Style",
    annotation_column="annotation",
    lipid_class_column="lipid_class",
    intensity_column="intensity",
    edge_colour_column="transformation",
    node_colors=None
):

    if node_colors is None:
        node_colors = NODE_COLORS

    p4c.create_visual_style(style_name)

    # ---------------------------------------------------------
    # Node colour
    # ---------------------------------------------------------

    p4c.set_node_color_mapping(
        table_column=lipid_class_column,
        mapping_type="d",
        table_column_values=list(node_colors.keys()),
        colors=list(node_colors.values()),
        style_name=style_name,
    )

    # ---------------------------------------------------------
    # Labels
    # ---------------------------------------------------------

    p4c.set_node_label_mapping(
        table_column=annotation_column,
        style_name=style_name
    )

    # ---------------------------------------------------------
    # Node size
    # ---------------------------------------------------------

    p4c.set_node_size_mapping(
        table_column=intensity_column,
        table_column_values=[1e4, 1e6],
        sizes=[20, 60],
        mapping_type="c",
        default_size=20,
        style_name=style_name
    )

    # ---------------------------------------------------------
    # Edge width
    # ---------------------------------------------------------

    p4c.set_edge_line_width_mapping(
        table_column="entropy_similarity",
        table_column_values=[0.75, 1.0],
        widths=[2, 8],
        mapping_type="c",
        default_width=2,
        style_name=style_name
    )

    # ---------------------------------------------------------
    # Edge colour
    # ---------------------------------------------------------

    p4c.set_edge_color_mapping(
        table_column=edge_colour_column,
        mapping_type="d",
        table_column_values=list(EDGE_COLORS.keys()),
        colors=list(EDGE_COLORS.values()),
        style_name=style_name,
    )

In [ ]:
# This version includes unidentified features
def build_marker_network(
    identified_matches,
    mz_tolerance=0.015,
    entropy_threshold=0.75,
):

# These transformations are those used by the Cytoscape app, MettaNetter_2
    KNOWN_TRANSFORMATIONS = {
        1.031634: "Aminotransferase",
        2.015650: "Hydrogenation (+H2)",
        14.00307401: "Tertiary Amine (+N)",
        14.015650: "CH2 addition (+CH2)",
        15.01089905: "Secondary Amine (+NH)",
        16.01872408: "Primary Amine (+NH2)",
        27.99491464: "+CO",
        28.031300: "Elongation (+C2H4)",
        56.062600: "Elongation (+C4H8)",
        15.994915: "Oxidation (+O)",
        31.989830: "Dioxidation (+2O)",
        47.984745: "Trioxidation (+3O)",
        18.010565: "Hydration (+H2O)",
        95.961248: "+HPO4",
    	42.010564: "+C2H2O",
	    42.021798: "+CH2N2",
    	42.034374: "+C2H4N",
        43.98982928: "+CO2",
    	44.01363872: "+CH2ON",
    	44.99765432: "+CHO2",
    	46.00548: "+CO2H2",
    	55.98982928: "C2O2",
    	57.02146376: "+C2H3NO",
    	57.03403983: "+C3H5O",
    	57.057849: "+C3H7N",
    	59.01330439: "+C2H3O2",
    	59.02453777: "+CH3N2O",
    	60.02113: "+C2H4O2",
    	61.9475268: "Pyrophosphate (+PP)",
    	62.000395: "+CH2O3",
    	67.05477526: "+C5H7",
    	71.03711384: "+C3H5NO",
        79.95681572: "+SO3",
    	79.96633236: "+HPO3",
    	80.964642: "+HSO3",
    	84.02112943: "+C4H4O2",
    	86.00039399: "+C3H2O3",
    	86.096974: "+C5H12N",
    	87.03202848: "+C3H5NO2",
    	97.05276391: "+C5H7NO",
    	99.06841398: "+C5H9NO",
    	101.0476785: "+C4H7NO2",
    	102.031695: "+C4H6O3",
    	103.0091856: "+C3H5NOS",
    	110.0354368: "+C4H4N3O",
    	111.0194524: "+C4H3N2O2",
    	113.0840641: "+C6H11NO",
    	114.0429275: "+C4H6N2O2",
    	114.079313: "+C5H10N2O",
    	115.0269431: "+C4H5NO3",
    	125.0351025: "+C5H5N2O2",
    	128.0585776: "+C5H8N2O2",
    	128.0949631: "+C6H12N2O",
	    128.131348: "Poly Amine Reaction B",
    	129.0425932: "+C5H7NO3",
    	131.0404858: "+C5H9NOS",
    	132.0422589: "+C5H8O4",
    	134.0466702: "+C5H4N5",
    	137.0589119: "+C6H7N3O",
    	147.068414: "+C9H9NO",
    	150.0415848: "+C5H4N5O",
    	156.1011111: "+C6H12N4O",
    	160.9404898: "+H3O6P2",
    	162.0528236: "+C6H10O5",
    	163.0633286: "+C9H9NO2",
    	169.998028: "+C3H7O6P",
    	176.0320881: "+C6H8O6",
    	178.0477382: "+C6H10O6",
    	186.079313: "+C11H10N2O",
    	222.0132859: "+C6H10N2O3S2",
    	224.079707: "+C10H12N2O4",
    	226.0589716: "+C9H10N2O5",
    	226.0775996: "+C10H14N2O2S",
    	229.0140109: "+C8H8NO5P",
    	238.2296658: "+C16H30O",
    	242.0191559: "+C6H11O8P",
    	243.0803393: "+C10H15N2O3S",
    	249.0861894: "+C10H11N5O3",
    	265.081104: "+C10H11N5O4",
    	281.271864: "+C18H35NO",
    	289.0732426: "+C10H15N3O5S",
    	304.0460394: "+C10H13N2O7P",
    	305.0412884: "+C9H12N3O7P",
    	306.0253039: "+C9H11N2O8P",
    	323.051855: "+C9H14N3O8P",
    	329.0525217: "+C10H12N5O6P",
    	340.1005618: "+C12H20O11",
    	345.0474364: "+C10H12N5O7P",
    	384.0123717: "+C10H14N2O10P2",
    	385.0076207: "+C9H13N3O10P2",
    	385.9916363: "+C9H12N2O11P2",
    	409.0188541: "+C10H13N5O9P2",
    	417.204592: "Poly Amine Reaction C",
    	425.0137687: "+C10H13N5O10P2",
    	486.1584707: "+C18H30O15",
    	617.098779: "+C17H30N7O10P3S",
    	692.08319: "+C19H31N6O14P3S",
    	705.29382: "+C27H47N9O9S2",
        721.109739: "+C20H34N7O14P3S",
    	749.104654: "+C21H34N7O15P3S",
    	751.120304: "Succinyl CoA Synthetase",
    	758.676152: "+C15H9I4NO3",
    	763.083919: "+C21H32N7O16P3S",
    	766.107394: "+C21H35N7O16P3S",
    }

    transformation_masses = np.array(
        list(KNOWN_TRANSFORMATIONS.keys()),
        dtype=float,
    )

    transformation_labels = list(
        KNOWN_TRANSFORMATIONS.values()
    )

    # =========================================================
    # Counters
    # =========================================================

    n_features = len(identified_matches)

    n_no_ms2 = 0
    n_no_msp = 0
    n_no_ms2_and_msp = 0

    n_candidate_pairs = 0
    n_similarity = 0
    n_above_threshold = 0
    n_edges = 0

    # =========================================================
    # Precompute m/z and RT
    # =========================================================

    mz = np.array(
        [f["precursor_mz"] for f in identified_matches],
        dtype=float,
    )

    rt = np.array(
        [f["precursor_rt"] for f in identified_matches],
        dtype=float,
    )

    # =========================================================
    # Prepare MS2 AND count missing data
    # =========================================================

    for feature in identified_matches:

        peaks = feature.get("cent_ms2_peaks")

        if peaks is None or len(peaks) == 0:

            feature["_ms2"] = None
            n_no_ms2 += 1

        else:

            feature["_ms2"] = np.asarray(
                peaks,
                dtype=np.float32,
            )

        # Count MSP matches separately
        if feature.get("MSP_match") is None:

            n_no_msp += 1

            if feature["_ms2"] is None:
                n_no_ms2_and_msp += 1

    # =========================================================
    # Create graph
    # =========================================================

    G = nx.MultiGraph()

    # =========================================================
    # Helper function for node information
    # =========================================================

    def add_feature_node(feature):

        node_id = (
            feature["file_id"],
            feature["peak_id"],
        )

        # Don't add the same node more than once
        if node_id in G:
            return node_id

        annotation = None

        if feature.get("MSP_match") is not None:
            annotation = feature["MSP_match"]["match"]["name"]

        G.add_node(
            node_id,
            feature=feature["feature_id"],
            cluster_id=feature["cluster_id"],
            mz=feature["precursor_mz"],
            rt=feature["precursor_rt"],
            intensity=feature["precursor_intensity"],
            classification=feature["classification"],
            lipid_class=feature["Class"],
            annotation=annotation,
        )

        return node_id

    # =========================================================
    # Build m/z index
    # =========================================================

    order = np.argsort(mz)
    mz_sorted = mz[order]

    # =========================================================
    # Find candidate pairs
    # =========================================================

    for i, feature in enumerate(identified_matches):

        # ---------------------------------------------
        # Skip features without MS2
        # ---------------------------------------------

        query = feature["_ms2"]

        if query is None:
            continue

        mz_i = mz[i]

        # =====================================================
        # Search each transformation
        # =====================================================

        for transformation_mass, transformation_label in zip(
            transformation_masses,
            transformation_labels,
        ):

            target_mz = mz_i + transformation_mass

            left = np.searchsorted(
                mz_sorted,
                target_mz - mz_tolerance,
                side="left",
            )

            right = np.searchsorted(
                mz_sorted,
                target_mz + mz_tolerance,
                side="right",
            )

            neighbours = order[left:right]

            # =================================================
            # Compare candidates
            # =================================================

            for j in neighbours:

                if j == i:
                    continue

                n_candidate_pairs += 1

                other = identified_matches[j]

                # ---------------------------------------------
                # Other feature must have MS2
                # ---------------------------------------------

                reference = other["_ms2"]

                if reference is None:
                    continue

                # =================================================
                # Entropy similarity
                # =================================================

                n_similarity += 1

                similarity = me.calculate_entropy_similarity(
                    query,
                    reference,
                )

                if similarity < entropy_threshold:
                    continue

                n_above_threshold += 1

                # =================================================
                # Valid connection found
                # =================================================

                node_id_i = add_feature_node(feature)
                node_id_j = add_feature_node(other)

                delta_mz = mz[j] - mz[i]

                G.add_edge(
                    node_id_i,
                    node_id_j,
                    entropy_similarity=similarity,
                    delta_mz=delta_mz,
                    delta_rt=abs(
                        rt[i] - rt[j]
                    ),
                    transformation=transformation_label,
                )

                n_edges += 1

    # =========================================================
    # Diagnostics
    # =========================================================

    print("\nMarker network statistics")
    print("-------------------------")
    print(f"Total input features:    {n_features:,}")
    print(f"No MS2:                  {n_no_ms2:,}")
    print(f"No MSP match:            {n_no_msp:,}")
    print(f"No MS2 AND no MSP:       {n_no_ms2_and_msp:,}")
    print(f"Candidate pairs:         {n_candidate_pairs:,}")
    print(f"Entropy comparisons:     {n_similarity:,}")
    print(f"Above entropy threshold: {n_above_threshold:,}")
    print(f"Edges created:           {n_edges:,}")
    print(f"Connected nodes:         {G.number_of_nodes():,}")

    return G

In [ ]:
G = build_marker_network(
    aggregated_matches,
    entropy_threshold=0.75
)

show_marker_network(G)